# Perfilamento da camada bronze

**Objetivo:** antes de escrever qualquer regra da silver, descobrir o que está errado, estranho ou ambíguo nos dados como o governo publicou. Cada descoberta (um *achado*) vira uma regra da silver ou uma decisão da gold.

**Tabelas investigadas:** `bronze.caged_movimentacao` (995.781 linhas), `bronze.rais_vinculo` (1.931.903) e `bronze.pnad_pessoa` (46.644).

**Como ler este notebook:** cada achado tem uma **pergunta**, as **consultas**, uma **interpretação** logo depois de cada resultado e uma **conclusão** com a decisão. O resumo de todos os achados fica em `docs/perfilamento.md`.

| # | Pergunta | Status |
|---|---|---|
| A0 | A estrutura das tabelas está certa? | ✅ corrigido na bronze |
| A1 | O que cada linha representa no tempo? | ✅ |
| A2 | Os números são números? | ✅ |
| A3 | O mesmo conceito tem o mesmo código nas três fontes? | ✅ |
| A4 | Quais valores querem dizer "não sei"? | ✅ |
| A5 | Os valores são plausíveis? | ✅ regra proposta, a revisar |
| A6 | O que a fonte não consegue responder? | ✅ |

---
## A0. A estrutura das tabelas está certa?

**Pergunta:** alguma linha não coube no schema da tabela?

O Auto Loader cria sozinho a coluna `_rescued_data`, uma "caixa de achados e perdidos" onde ele guarda os valores que não encaixaram no schema. Ela é o **alarme de problemas estruturais**, e o normal é estar vazia.

In [0]:
%sql
SELECT 'CAGED' AS tabela, COUNT(*) FILTER (WHERE _rescued_data IS NOT NULL) AS linhas_resgatadas FROM brazil_employment.bronze.caged_movimentacao
UNION ALL SELECT 'RAIS', COUNT(*) FILTER (WHERE _rescued_data IS NOT NULL) FROM brazil_employment.bronze.rais_vinculo
UNION ALL SELECT 'PNAD', COUNT(*) FILTER (WHERE _rescued_data IS NOT NULL) FROM brazil_employment.bronze.pnad_pessoa;

**Interpretação:** 0 linhas resgatadas nas três tabelas. O alarme está limpo.

Mas nem sempre foi assim. Na primeira carga, a PNAD tinha **46.644 linhas resgatadas (100%)**. Esta consulta é a **validação da correção**.

### Conclusão A0
- **O que vi:** todas as linhas da PNAD tinham `_rescued_data` preenchida; CAGED e RAIS, nenhuma.
- **Causa:** `ano` e `trimestre` estavam em dois lugares: dentro do arquivo (texto) e no nome das pastas (número). O Auto Loader ficou com o valor das pastas e jogou o do arquivo no `_rescued_data`.
- **Por que importa:** um alarme que dispara em 100% das linhas deixa passar um problema real, como uma mudança de formato do IBGE.
- **Decisão:** corrigido já na bronze, com `partitionColumns = ""` na PNAD e full refresh da tabela. Regra geral: **cada informação deve ter uma fonte só**.

---
## A1. O que cada linha representa no tempo?

**Pergunta:** os dados de um arquivo de janeiro são todos de janeiro?

O CAGED tem duas colunas de mês que parecem a mesma coisa:
- `competencia`: o mês da **divulgação** (a pasta de onde o arquivo veio);
- `competenciamov`: o mês em que a admissão ou o desligamento **aconteceu**.

A coluna `tipo` separa os três arquivos mensais: **MOV** (declarado no prazo), **FOR** (fora do prazo) e **EXC** (exclusões).

### Consulta 1: o que vem dentro da divulgação de janeiro/2026

In [0]:
%sql
SELECT tipo, competenciamov, COUNT(*) AS linhas
FROM brazil_employment.bronze.caged_movimentacao
WHERE competencia = 202601
GROUP BY ALL
ORDER BY tipo, competenciamov;

**Interpretação:**
- **MOV** tem só `202601` (80.125 linhas): o que as empresas declararam no prazo, sobre o próprio mês.
- **FOR** vai de `202501` a `202512` (906 linhas, 535 delas de dezembro): declarações **atrasadas**, como uma admissão de março/2025 informada só agora.
- **EXC** vai de `202009` a `202512` (192 linhas): **cancelamentos**, inclusive de admissões declaradas errado em 2020.

Ou seja: **a divulgação de janeiro mexe em meses de cinco anos atrás.**

### Consulta 2: visão geral das 12 divulgações

In [0]:
%sql
SELECT
  tipo,
  MIN(competenciamov)            AS mes_mais_antigo,
  MAX(competenciamov)            AS mes_mais_recente,
  COUNT(DISTINCT competenciamov) AS meses_afetados,
  SUM(CASE WHEN competenciamov < '202507' THEN 1 ELSE 0 END) AS linhas_antes_da_janela
FROM brazil_employment.bronze.caged_movimentacao
GROUP BY tipo
ORDER BY tipo;

**Interpretação:**
- Os arquivos **EXC** alteram **69 meses diferentes**, desde jan/2020.
- Os arquivos **FOR** alteram **23 meses**, desde jul/2024.
- São **4.133 linhas** (3.070 + 1.063) de meses **anteriores** à janela do projeto (07/2025 a 06/2026).
- A comparação `competenciamov < '202507'` usa texto (com aspas) porque, na bronze, a coluna ainda é texto. Funciona porque AAAAMM em texto ordena igual a número.

### Conclusão A1
- **O que vi:** cada divulgação mensal corrige meses passados, alguns de 2020.
- **Por que importa:** somar pelo mês da divulgação infla o saldo do mês atual com fatos antigos; e um mês com 2 movimentações isoladas parece uma queda dramática do emprego.
- **Decisões:**
  1. o saldo é agrupado por `competenciamov`, nunca por `competencia`;
  2. a silver é uma **materialized view**, que recalcula quando chega arquivo novo;
  3. a gold tem um **filtro de período explícito**.

---
## A2. Os números são números?

**Pergunta:** as colunas que vão entrar em somas e médias podem ser usadas em contas?

Na bronze, tudo é texto, de propósito. A questão é em que formato esse texto está.

### 2a. Valores reais

In [0]:
%sql
SELECT salario, horascontratuais FROM brazil_employment.bronze.caged_movimentacao LIMIT 3;

**Interpretação:** no CAGED, o salário e as horas usam **vírgula** como separador decimal (`937,00`, `36,00`).

In [0]:
%sql
SELECT vl_rem_media_nom FROM brazil_employment.bronze.rais_vinculo LIMIT 3;

**Interpretação:** na RAIS, as 3 primeiras linhas vieram com `.00`, mas isso é coincidência do `LIMIT 3`: dá a falsa impressão de que toda remuneração é zero. A consulta abaixo mostra também um valor preenchido.

In [0]:
%sql
SELECT vl_rem_media_nom
FROM brazil_employment.bronze.rais_vinculo
WHERE vl_rem_media_nom <> '.00'
LIMIT 3;

**Interpretação:** quando preenchida, a remuneração da RAIS usa **ponto** decimal (valores como `2239.74` e `7000.00`), e o zero é escrito como `.00`. **Convenção diferente da do CAGED**, que usa vírgula. (Os exemplos mudam a cada execução: `LIMIT` sem `ORDER BY` pega quaisquer 3 linhas.)

In [0]:
%sql
SELECT peso, idade FROM brazil_employment.bronze.pnad_pessoa LIMIT 3;

**Interpretação:** na PNAD, os números vêm com **zeros à esquerda**, herança do arquivo de largura fixa do IBGE: peso `000786.34722912`, idade `070`.

### 2b. A conversão ingênua

In [0]:
%sql
SELECT CAST(salario AS DOUBLE) FROM brazil_employment.bronze.caged_movimentacao LIMIT 5;

**Interpretação:** o `CAST` direto **dá erro** (`NumberFormatException`), porque `937,00` não é um número válido pra máquina. Se a silver fizesse isso, o pipeline inteiro pararia.

### 2c. A conversão tolerante

In [0]:
%sql
SELECT
  COUNT(*)                                              AS total,
  COUNT(try_cast(salario AS DOUBLE))                    AS convertidos_sem_tratar,
  COUNT(try_cast(replace(salario, ',', '.') AS DOUBLE)) AS convertidos_trocando_virgula
FROM brazil_employment.bronze.caged_movimentacao;

**Interpretação:**
- `convertidos_sem_tratar` = **0** de 995.781. O `try_cast` não dá erro: ele devolve `NULL` **em silêncio**. Todos os salários sumiriam e a média sairia vazia, sem nenhum alerta.
- `convertidos_trocando_virgula` = **995.781** (100%). Trocando a vírgula por ponto, tudo converte.

### Conclusão A2
- **O que vi:** cada fonte escreve números de um jeito: vírgula (CAGED), ponto e `.00` (RAIS), zeros à esquerda (PNAD).
- **Por que importa:** o `CAST` direto quebra o pipeline; o `try_cast` sem tratamento apaga os valores em silêncio.
- **Decisão:** na silver, uma regra de conversão **específica por fonte e por coluna**, e uma *expectation* que mede quantos valores não converteram. Se esse número subir num mês, o formato da fonte mudou.

---
## A3. O mesmo conceito tem o mesmo código nas três fontes?

**Pergunta:** "mulher" e "parda" são escritos do mesmo jeito no CAGED, na RAIS e na PNAD?

Sexo e raça/cor existem nas três fontes e são justamente os cortes que o painel vai fazer. Os nomes das colunas mudam: `sexo`/`racacor` no CAGED, `sexo_codigo`/`raca_cor_codigo` na RAIS e `sexo`/`cor_raca` na PNAD.

### Sexo

In [0]:
%sql
SELECT 'CAGED' AS fonte, sexo AS codigo, COUNT(*) AS n
FROM brazil_employment.bronze.caged_movimentacao WHERE competencia = 202601 AND tipo = 'MOV' GROUP BY ALL
UNION ALL
SELECT 'RAIS', sexo_codigo, COUNT(*) FROM brazil_employment.bronze.rais_vinculo GROUP BY ALL
UNION ALL
SELECT 'PNAD', sexo, COUNT(*) FROM brazil_employment.bronze.pnad_pessoa WHERE ano = '2026' AND trimestre = '2' GROUP BY ALL
ORDER BY 1, 2;

**Interpretação:** o CAGED usa os códigos **1 e 3**; a RAIS e a PNAD usam **1 e 2**. Pelo dicionário oficial de cada fonte, **mulher é 3 no CAGED e 2 nas outras duas**.

### Raça/cor

In [0]:
%sql
SELECT 'CAGED' AS fonte, racacor AS codigo, COUNT(*) AS n
FROM brazil_employment.bronze.caged_movimentacao WHERE competencia = 202601 AND tipo = 'MOV' GROUP BY ALL
UNION ALL
SELECT 'RAIS', raca_cor_codigo, COUNT(*) FROM brazil_employment.bronze.rais_vinculo GROUP BY ALL
UNION ALL
SELECT 'PNAD', cor_raca, COUNT(*) FROM brazil_employment.bronze.pnad_pessoa WHERE ano = '2026' AND trimestre = '2' GROUP BY ALL
ORDER BY 1, 3 DESC;

**Interpretação:** o código mais frequente em cada fonte é diferente: **3** no CAGED (62.540), **8** na RAIS (1.456.849) e **4** na PNAD (5.339). Nos três casos, é **"parda"**, a maioria da população da RMF.

| Conceito | CAGED | RAIS | PNAD |
|---|---|---|---|
| Mulher | **3** | 2 | 2 |
| Parda | **3** | **8** | **4** |
| Branca | 1 | 2 | 1 |
| Preta | 2 | 4 | 2 |
| Não identificado | 9 (e 6 = não informada) | 9 (e −1 = ignorado) | 9 |

### Conclusão A3
- **O que vi:** cada fonte codifica sexo e raça/cor de um jeito.
- **Por que importa:** se a gold juntar as fontes pelo código, **as mulheres somem do CAGED** e **as pessoas pardas da PNAD viram amarelas** (o código 3 é amarela na PNAD). Nenhum erro aparece: só números plausíveis e errados. É o achado mais importante do projeto.
- **Decisão:** na silver, cada fonte traduz os seus códigos para **rótulos comuns** (`'Mulher'`, `'Parda'`), mantendo o código original ao lado (**dimensão conformada**). Um teste automatizado trava `SEXO_CAGED["3"] == "Mulher"`.

---
## A4. Quais valores querem dizer "não sei"?

**Pergunta:** quais valores parecem dados mas significam "não informado" ou "não se aplica"?

As fontes usam códigos especiais (`9`, `99`, `999`, `.00`) ou nulos pra dizer que a informação não existe. Nem todo código estranho é "não sei": às vezes é um valor válido fora da sequência.

In [0]:
%sql
SELECT 'categoria = 999'                         AS o_que, COUNT(*) AS n FROM brazil_employment.bronze.caged_movimentacao WHERE competencia = 202601 AND tipo = 'MOV' AND categoria = '999'
UNION ALL SELECT 'graudeinstrucao = 80',          COUNT(*) FROM brazil_employment.bronze.caged_movimentacao WHERE competencia = 202601 AND tipo = 'MOV' AND graudeinstrucao = '80'
UNION ALL SELECT 'RAIS remuneração = .00',        COUNT(*) FROM brazil_employment.bronze.rais_vinculo WHERE vl_rem_media_nom = '.00'
UNION ALL SELECT 'PNAD sem condição na força de trabalho', COUNT(*) FROM brazil_employment.bronze.pnad_pessoa WHERE ano = '2026' AND trimestre = '2' AND condicao_forca_trabalho IS NULL
UNION ALL SELECT '... desses, com menos de 14 anos', COUNT(*) FROM brazil_employment.bronze.pnad_pessoa WHERE ano = '2026' AND trimestre = '2' AND condicao_forca_trabalho IS NULL AND CAST(idade AS INT) < 14;

**Interpretação, linha a linha:**
- **`categoria = 999` (34):** é "Não identificado" no dicionário. Como valor, viraria uma barra "999" no gráfico.
- **`graudeinstrucao = 80` (1.005):** **não é erro**. É "Pós-graduação completa", só está fora da sequência 1–11. Sem o dicionário, eu apagaria um dado válido.
- **RAIS remuneração `.00` (202.672):** vínculos sem remuneração informada. Como zero, puxariam a remuneração média pra baixo.
- **PNAD sem condição na força de trabalho (1.374) e, desses, com menos de 14 anos (1.374):** os dois números são **iguais**. Todo mundo sem essa informação tem menos de 14 anos. Aqui o nulo **não é dado faltando**: quer dizer **"não se aplica"**.

### Conclusão A4
- **O que vi:** códigos de "não identificado" no CAGED, `.00` na RAIS e nulos com significado na PNAD.
- **Por que importa:** "não sei" tratado como valor vira categoria falsa ou distorce médias; e transformar o nulo da PNAD em 0 erraria o denominador da taxa de desocupação.
- **Decisão:** na silver, os códigos de "não identificado" viram `NULL` nas colunas traduzidas, e o `.00` da RAIS também. Na PNAD, o `NULL` **continua `NULL`**, com o significado documentado no `COMMENT` da coluna.

---
## A5. Os valores são plausíveis?

**Pergunta:** o salário informado no CAGED dá pra comparar diretamente entre as pessoas?

O salário de admissão é um dos indicadores mais pedidos ("quanto ganha um jovem no primeiro emprego formal?"). O CAGED informa o valor e, separado, a **unidade** (`unidadesalariocodigo`): 5 = mês, 1 = hora, 2 = dia, 3 = semana, 4 = quinzena, 6 = tarefa, 7 = variável, 99 = não identificado.

O `p1_p50_p99` mostra três percentis: o valor abaixo do qual estão 1%, 50% (a **mediana**) e 99% dos salários. Percentis não se deixam distorcer por extremos, ao contrário da média.

### 5a. Distribuição por unidade

In [0]:
%sql
SELECT
  unidadesalariocodigo,
  COUNT(*) AS n,
  percentile_approx(try_cast(replace(salario, ',', '.') AS DOUBLE), array(0.01, 0.5, 0.99)) AS p1_p50_p99,
  ROUND(AVG(try_cast(replace(salario, ',', '.') AS DOUBLE)))                                AS media
FROM brazil_employment.bronze.caged_movimentacao
WHERE competencia = 202601 AND tipo = 'MOV'
GROUP BY 1
ORDER BY n DESC;

**Interpretação:**

| unidade | n | p1 | mediana | p99 | média |
|---|---|---|---|---|---|
| **5 (mês)** | 75.760 | **7,61** | 1.658 | 7.188 | 2.040 |
| **1 (hora)** | 3.143 | 0 | **1.113** | 393.229 | 18.381 |
| 99 (não identificado) | 470 | 0 | 0 | 2.230 | 150 |

Duas coisas não fazem sentido:
- **Salários "mensais" de R$ 7,61** (o 1% mais baixo da unidade 5). Ninguém ganha R$ 7 por mês com carteira assinada.
- **Salários "por hora" com mediana de R$ 1.113.** Ninguém ganha R$ 1.113 por hora numa admissão típica.

Ou seja: **a unidade informada não é confiável.** Há valores por hora marcados como mensais, e valores mensais marcados como por hora.

### 5b. Os extremos

In [0]:
%sql
SELECT
  COUNT(*) FILTER (WHERE s = 0)             AS zero,
  COUNT(*) FILTER (WHERE s > 0 AND s < 100) AS abaixo_de_100,
  COUNT(*) FILTER (WHERE s > 50000)         AS acima_de_50mil,
  MAX(s)                                    AS maximo
FROM (
  SELECT try_cast(replace(salario, ',', '.') AS DOUBLE) AS s
  FROM brazil_employment.bronze.caged_movimentacao
  WHERE competencia = 202601 AND tipo = 'MOV'
);

**Interpretação:** 1.127 admissões com salário **zero**, 789 entre R$ 0 e R$ 100, 206 acima de R$ 50 mil e um máximo de **R$ 1.362.163**. Os extremos puxam a média (R$ 2.040) bem acima da mediana (R$ 1.658).

### 5c. Investigando a causa: qual é a referência?

In [0]:
%sql
SELECT salario, COUNT(*) AS n
FROM brazil_employment.bronze.caged_movimentacao
WHERE competencia = 202601 AND tipo = 'MOV'
GROUP BY 1
ORDER BY n DESC
LIMIT 5;

**Interpretação:** o valor mais frequente é **R$ 1.621,00** (21.986 admissões, 27% do mês), compatível com o **salário mínimo de 2026**. Ele vira a régua natural pra julgar se um valor é plausível. (Confira o valor oficial do salário mínimo de 2026 antes de fixá-lo no código.)

E o mistério dos R$ 7: **R$ 7,37 × 220 horas mensais = R$ 1.621**. O "salário mensal" de R$ 7,37 é o **salário mínimo por hora**, informado com a unidade errada.

### 5d. Onde cai cada salário "mensal", em salários mínimos

In [0]:
%sql
SELECT
  CASE
    WHEN s = 0               THEN '1. zero'
    WHEN s < 0.1 * 1621      THEN '2. abaixo de 0,1 SM'
    WHEN s < 0.3 * 1621      THEN '3. 0,1 a 0,3 SM'
    WHEN s < 0.5 * 1621      THEN '4. 0,3 a 0,5 SM'
    WHEN s <= 30 * 1621      THEN '5. 0,5 a 30 SM'
    ELSE                          '6. acima de 30 SM'
  END AS faixa,
  COUNT(*)                                                      AS n,
  ROUND(AVG(h), 1)                                              AS horas_semanais_media,
  SUM(CASE WHEN indtrabparcial = '1' THEN 1 ELSE 0 END)         AS tempo_parcial
FROM (
  SELECT *,
         try_cast(replace(salario, ',', '.') AS DOUBLE)          AS s,
         try_cast(replace(horascontratuais, ',', '.') AS DOUBLE) AS h
  FROM brazil_employment.bronze.caged_movimentacao
  WHERE competencia = 202601 AND tipo = 'MOV' AND unidadesalariocodigo = '5'
)
GROUP BY 1
ORDER BY 1;

**Interpretação:**

| faixa | n | horas/semana | tempo parcial |
|---|---|---|---|
| zero | 184 | 36,4 | 44 |
| abaixo de 0,1 SM | 736 | 32,2 | 381 |
| 0,1 a 0,3 SM | 37 | 26,1 | 15 |
| **0,3 a 0,5 SM** | 723 | **21,5** | **306** |
| **0,5 a 30 SM** | **74.050** | 42,7 | 1.525 |
| acima de 30 SM | 30 | 44,0 | 0 |

- **Abaixo de 0,1 SM** (736): os valores por hora com a unidade errada, como os R$ 7,37.
- **0,3 a 0,5 SM** (723): **não é erro**. São contratos de **tempo parcial**, com média de 21,5 horas por semana (contra 42,7 no geral) e 306 marcados como parciais. Um corte em "1 salário mínimo" apagaria esses trabalhadores reais.
- **0,5 a 30 SM:** 97,7% das admissões mensais, o grosso legítimo.

### 5e. Quanto a regra muda o resultado?

In [0]:
%sql
SELECT
  COUNT(CASE WHEN s > 0 THEN 1 END)                                   AS n_regra_simples,
  ROUND(percentile_approx(CASE WHEN s > 0 THEN s END, 0.5))          AS mediana_regra_simples,
  ROUND(AVG(CASE WHEN s > 0 THEN s END))                             AS media_regra_simples,
  COUNT(CASE WHEN s BETWEEN 0.3 * 1621 AND 30 * 1621 THEN 1 END)      AS n_regra_plausivel,
  ROUND(percentile_approx(CASE WHEN s BETWEEN 0.3 * 1621 AND 30 * 1621 THEN s END, 0.5)) AS mediana_regra_plausivel,
  ROUND(AVG(CASE WHEN s BETWEEN 0.3 * 1621 AND 30 * 1621 THEN s END)) AS media_regra_plausivel
FROM (
  SELECT try_cast(replace(salario, ',', '.') AS DOUBLE) AS s
  FROM brazil_employment.bronze.caged_movimentacao
  WHERE competencia = 202601 AND tipo = 'MOV' AND unidadesalariocodigo = '5'
);

**Interpretação:**

| | regra simples (mês e > 0) | regra plausível (mês, entre 0,3 e 30 SM) |
|---|---|---|
| admissões consideradas | 75.576 | 74.773 (98,9%) |
| **mediana** | **R$ 1.658** | **R$ 1.658** |
| média | R$ 2.045 | R$ 2.010 |

A regra plausível descarta só 1,1% das admissões, e **a mediana não muda**. É a prova de que a mediana resiste a valores errados, e a média não. A regra limpa as caudas sem distorcer o centro.

### Conclusão A5 (regra proposta, a revisar)
- **O que vi:** a unidade do salário não é confiável. Há valores por hora marcados como mensais (R$ 7,37 = salário mínimo por hora), valores mensais marcados como por hora, salários zero e extremos acima de R$ 1 milhão.
- **Por que importa:** comparar salários sem tratar mistura unidades; a média fica inflada pelos extremos.
- **Decisão proposta para a silver:** criar `salario_mensal` = valor informado **somente quando** a unidade é 5 (mês) **e** o valor está entre **0,3 e 30 salários mínimos**. Fora disso, `NULL`. O valor e a unidade originais ficam em colunas ao lado. O piso de 0,3 SM preserva os contratos de tempo parcial; o teto de 30 SM corta os erros de digitação evidentes.
- **Alternativa considerada e descartada:** converter os valores por hora pra mês (× 220). Seria adivinhar a jornada de cada pessoa, e o ganho é pequeno (~1% das admissões).
- **Decisão para a gold:** salário de admissão sempre pela **mediana**, nunca pela média.

---
## A6. O que a fonte não consegue responder?

**Pergunta:** quais perguntas parecem possíveis mas não são, e quais limites cada fonte tem?

Antes de desenhar a gold, é preciso saber o que **não** dá pra fazer. Cada consulta testa um limite diferente.

### 6a. O tipo de admissão (CAGED)

In [0]:
%sql
SELECT tipomovimentacao, COUNT(*) AS n
FROM brazil_employment.bronze.caged_movimentacao
WHERE competencia = 202601 AND tipo = 'MOV' AND saldomovimentacao = '1'
GROUP BY 1
ORDER BY n DESC;

**Interpretação:** `97` = 39.649 e `35` = 11. Pelo dicionário, **97 é "Admissão de tipo ignorado"**. Em 99,97% das admissões, o CAGED público não diz se foi primeiro emprego, reemprego ou contrato por prazo determinado.

**Consequência:** "quantos jovens tiveram o **primeiro emprego**?" parece uma ótima pergunta pro secretário, mas **não tem resposta** com estes dados.

### 6b. Colunas sem informação (RAIS)

In [0]:
%sql
SELECT
  COUNT(DISTINCT bairros_fortaleza_codigo) AS bairros_fortaleza,
  COUNT(DISTINCT ano_chegada_brasil)       AS ano_chegada,
  ROUND(100 * AVG(CASE WHEN municipio_trab_codigo = '999999' THEN 1 ELSE 0 END), 1) AS pct_municipio_trab_vazio
FROM brazil_employment.bronze.rais_vinculo;

**Interpretação:** 1 · 1 · 99,8%. Os bairros de Fortaleza têm **um único valor** (999997) em todas as linhas; o ano de chegada ao Brasil também; e o município de trabalho está vazio em 99,8%.

**Consequência:** a metodologia do Emprega+ citava os bairros de Fortaleza na RAIS como oportunidade, mas **eles não vêm preenchidos nos microdados públicos**. Colunas com um único valor não informam nada.

### 6c. Vínculos ativos e encerrados (RAIS)

In [0]:
%sql
SELECT ind_vinculo_ativo_31_12_codigo, COUNT(*) AS n
FROM brazil_employment.bronze.rais_vinculo
GROUP BY 1;

**Interpretação:** `1` = 1.345.884 vínculos **ativos em 31/12** e `0` = 586.019 **encerrados** durante o ano.

**Consequência:** a RAIS guarda todos os vínculos que passaram pelo ano. Contar as 1,93 milhão de linhas como "empregos" superestimaria o estoque em 44%.

### 6d. O tamanho da amostra (PNAD)

In [0]:
%sql
SELECT
  ano, trimestre,
  COUNT(*)                                        AS pessoas_entrevistadas,
  COUNT(*) FILTER (WHERE condicao_ocupacao = '2') AS desocupados_na_amostra,
  ROUND(SUM(CAST(peso AS DOUBLE)))                AS populacao_representada
FROM brazil_employment.bronze.pnad_pessoa
GROUP BY ALL
ORDER BY 1, 2;

**Interpretação:** de **7.177 a 8.055 entrevistados** por trimestre, representando cerca de **4 milhões de pessoas** (soma dos pesos). Mas só **189 a 284 desocupados** aparecem na amostra.

**Consequência:** cada entrevistado "vale" uns 500 moradores. Com ~250 desocupados na amostra, quebrar a taxa de desocupação por município ou faixa etária fina geraria estimativas sem confiabilidade, que nem o IBGE divulgaria. E qualquer número da PNAD precisa somar o **peso**, nunca contar linhas.

### Conclusão A6
- **O que vi:** o tipo de admissão quase nunca vem preenchido; colunas da RAIS sem informação; a RAIS mistura vínculos ativos e encerrados; a amostra da PNAD é pequena para cortes finos.
- **Por que importa:** evita prometer ao secretário indicadores impossíveis ("primeiro emprego", "por bairro") e evita números inflados ou sem confiabilidade.
- **Decisões:**
  - **silver:** as colunas constantes da RAIS saem; entra uma coluna booleana `ativo_31_12`;
  - **gold:** "estoque de emprego" conta só `ativo_31_12`; a PNAD só no nível da RM inteira (ou cortes grandes, como sexo), sempre somando o peso; nada de "primeiro emprego";
  - **demo:** os limites entram na fala ao secretário, como transparência.

---
## Próximo passo

Resumir os achados A0 a A6 em `docs/perfilamento.md`, que vira a lista de tarefas da silver (fase 09).